In [ ]:
import pandas as pd
import numpy as np
import pickle

#import google cloud library
from google.cloud import bigquery
from google.cloud import storage
from google.cloud import aiplatform

#import library requirements
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier

# **Load Model**

In [ ]:
# Load the trained model and preprocessing pipeline using pickle
filename = 'final_model_gbc_bank_marketing_campaign.sav'
with open(filename, 'rb') as file:
    loaded_model = pickle.load(file)

# **Create Observation**

In [ ]:
new_cust = pd.DataFrame({
    'age': [56,24],
    'housing' : ['no','yes'],
    'loan' : ['no','yes'],
    'contact' : ['unknown','cellular'],
    'balance' : [1000,488],
    'job' : ['admin','management'],
    'month' : ['jan','apr'],
    'campaign' : [2,1],
    'pdays' : [-1,-1],
    'poutcome' : ['failure','success']
    })
new_cust

In [ ]:
#new_cust2 = pd.DataFrame({
    'age': [24],
    'housing' : ['no'],
    'loan' : ['no'],
    'contact' : ['cellular'],
    'balance' : [448],
    'job' : ['management'],
    'month' : ['apr'],
    'campaign' : [1],
    'pdays' : [-1],
    'poutcome' : ['unknown']
    })
new_cust2

In [ ]:
df = pd.read_csv("data_bank_marketing_campaign.csv")
df.head()

In [ ]:
# Separate the features and the target variable
X = df.drop('deposit', axis=1)
y = df['deposit']

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.2,
    random_state=0
)

# Define the preprocessing steps with handle_unknown='ignore'
transformer = ColumnTransformer([
    ('onehot', OneHotEncoder(handle_unknown='ignore'), ['job', 'housing', 'loan', 'contact', 'month', 'poutcome'])],
    remainder='passthrough'
)

# Create a pipeline that includes the transformer, scaler, and model
pipeline = Pipeline(steps=[
    ('preprocessor', transformer),
    ('scaler', StandardScaler(with_mean=False)),
    ('classifier', GradientBoostingClassifier(random_state=0))
])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

# Save the entire pipeline (including the preprocessor and scaler)
filename = 'final_model_gbc_bank_marketing_campaign.sav'
with open(filename, 'wb') as file:
    pickle.dump(pipeline, file)

print("Model and preprocessing pipeline saved to 'final_model_gbc_bank_marketing_campaign.sav'.")

## **Upload GCS**

In [ ]:
import os
## set up authentication using services account 
# Authenticate using service account
# os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = "../zaki-013/sa-development.json"
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = "../zaki-013/sa-development.json"

In [ ]:
# Load data from Google Biquery to local machine

project_id = 'dti-ds'
dataset_id = 'zaki_dataset_013'
table_id = 'x_bank_marketing_campaign'
region = 'us-central1'
bucket_name = 'zaki_gcs_013'
blob_name = 'x_bank_marketing_campaign.csv'
model_name = 'final_model_gbc_bank_marketing_campaign.sav'

In [ ]:
# Upload the model to Google Cloud Storage
try : 
    storage_client = storage.Client(project='dti-ds')
    bucket = storage_client.get_bucket('zaki_gcs_013') # Add bucket name
    blob_model = bucket.blob('Bank-Marketing-Campaign/final_model_gbc_bank_marketing_campaign.sav')
    blob_model.upload_from_filename('final_model_gbc_bank_marketing_campaign.sav')
  
    print ("Uploading model succeeded")
except:
    raise TypeError("An exception occurred")

# **Load Data From BigQuerry**

In [ ]:
#load data from BQ
from google.cloud import bigquery
## using bigquery client 
client = bigquery.Client(project=project_id)

# query 
query_job = client.query(f"""select * from {dataset_id}.{table_id}""")
df_cloud= query_job.result().to_dataframe()

df_cloud = df_cloud.loc[:, new_cust.columns]
df_cloud

In [ ]:
df_cloud.info()

In [ ]:
df_cloud['housing'] = df_cloud['housing'].astype(str).map({'True': 'yes', 'False': 'no'})


In [ ]:
df_cloud['loan'] = df_cloud['loan'].astype(str).map({'True': 'yes', 'False': 'no'})

In [ ]:
df_cloud.head()

In [ ]:
df_cloud.info()

In [ ]:
new_cust

In [ ]:
# Load the model
with open('final_model_gbc_bank_marketing_campaign.sav', 'rb') as f:
    loaded_model = pickle.load(f)

y_pred_cloud_new_data = loaded_model.predict(new_cust)
y_pred_cloud_new_data

In [ ]:
# data observation predicted
new_test_cloud = new_cust.copy()
new_test_cloud['open deposit'] = y_pred_cloud_new_data
new_test_cloud

In [ ]:
# loaded_model.predict(X_test.iloc[3:13])
y_pred_file_cloud = loaded_model.predict(df_cloud)
y_pred_file_cloud[:12]

In [ ]:
df_cloud['open deposit'] = y_pred_file_cloud
df_cloud

****

In [ ]:
table_id ='x_unseen_bank_marketing_campaign_predicted'

#construct a BiqQuery client object
client = bigquery.Client()

#Define the full table ID
table_full_id = f"{client.project}.{dataset_id}.{table_id}"

df_cloud.columns = ['_'.join(i.split(' ')) for i in df_cloud.columns]

# Load the DtaFrame into the BigQuerry table
job = client.load_table_from_dataframe(df_cloud,table_full_id)

job.result()  
print(f"Loaded {job.output_rows} rows into {table_full_id}")
